# 프로젝트: KoChatGPT 업그레이드 하기

# 1. Data 준비

아래 깃허브 레포지토리를 clone해주세요.
```
$ git clone https://github.com/airobotlab/KoChatGPT
# $ cp -r /content/KoChatGPT/colossalai_ChatGPT_230319/chatgpt /content/chatgpt

# 수정:
$ cp -r KoChatGPT/colossalai_ChatGPT_230319/chatgpt chatgpt
```

또는 터미널에서 진행

```# !git clone https://github.com/airobotlab/KoChatGPT
# !cp -r KoChatGPT/colossalai_ChatGPT_230319/chatgpt chatgpt1
```

In [ ]:
!pip install datasets
!pip install loralib
!pip install trl
!pip install accelerate
!pip install transformers  
# !pip install transformers==4.40.0  # 현재 version 5.3에서 한글 깨짐 -> 4.40.0 사용 -> 원복

##### 현재 코랩 버전에서는 일부 라이브러리를 사용하는데 장애가 있어 원본 소스 코드의 일부를 수정한 뒤 진행하겠습니다. 다음 코드를 위 clone 이후 동작시켜야합니다.

In [ ]:
# mv /home/jovyan/work/chatgpt . 로 현재 dir 이동후 진행

import os

modifications = [
    {
        "file": "chatgpt/trainer/callbacks/save_checkpoint.py",
        "changes": [
            {"line": 3, "old": "from chatgpt.trainer.strategies import ColossalAIStrategy, Strategy",
             "new": "from chatgpt.trainer.strategies import Strategy"},
            {"line": 71, "old": "only_rank0 = not isinstance(self.strategy, ColossalAIStrategy)",
             "new": "            only_rank0 = not isinstance(self.strategy)"},
        ],
    },
    {
        "file": "chatgpt/trainer/strategies/__init__.py",
        "changes": [
            {"line": 1, "old": "from .colossalai import ColossalAIStrategy", "new": ""},  # 삭제
            {"line": 5, "old": "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy', 'ColossalAIStrategy']",
             "new": "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy']"},
        ],
    },
    {
        "file": "chatgpt/dataset/reward_dataset.py",
        "changes": [
            {"line": 3, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ],
    },
    {
        "file": "chatgpt/trainer/base.py",
        "changes": [
            {"line": 8, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ]
    },
    {
        "file": "chatgpt/trainer/rm.py",
        "changes": [
            {"line": 8, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ]
    }
]


def modify_file(file_path, changes):
    """파일에서 지정된 줄을 찾아 내용을 수정하는 함수"""

    if not os.path.exists(file_path):
        print(f"⚠️ 파일이 존재하지 않습니다: {file_path}")
        return

    with open(file_path, "r", encoding="utf-8") as file:
        lines = file.readlines()

    modified = False

    for change in changes:
        line_index = change["line"]
        if 0 <= line_index < len(lines):
            if lines[line_index].strip() == change["old"]:
                lines[line_index] = change["new"] + "\n"
                modified = True
            else:
                print(f"⚠️ {file_path} 파일의 {change['line']}번째 줄이 예상과 다릅니다.")
                print(f"   예상: {change['old']}")
                print(f"   실제: {lines[line_index].strip()}")

    if modified:
        with open(file_path, "w", encoding="utf-8") as file:
            file.writelines(lines)
        print(f"✅ 수정 완료: {file_path}")
    else:
        print(f"⚠️ {file_path} 수정할 내용이 없습니다.")

for mod in modifications:
    modify_file(mod["file"], mod["changes"])

##### 소스 코드는 잘 불러와지는지, 필수 requirement들은 잘 설치되어 있는지 확인

In [34]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
import numpy

print("Torch version:{}".format(torch.__version__)) # Torch version:1.12.1
print("Cuda version: {}".format(torch.version.cuda)) # Cuda version: 11.3
print("transformers version: {}".format(transformers.__version__)) # transformers 4.28.0
print("GPU 사용 가능여부: {}".format(torch.cuda.is_available()))

# 만일 아래 모듈이 불러와지지 않는다면 Clone 및 수정을 잘 진행했는지 확인해주세요.
from chatgpt.trainer.strategies import NaiveStrategy

Torch version:2.7.1+cu118
Cuda version: 11.8
transformers version: 5.3.0
GPU 사용 가능여부: True


In [ ]:
# gpu 확인

# import torch
# print(f"PyTorch 버전: {torch.__version__}") # 2.6.0 이상이어야 함
# print(f"CUDA 사용 가능 여부: {torch.cuda.is_available()}") # True가 나와야 함
# if torch.cuda.is_available():
#     print(f"GPU 이름: {torch.cuda.get_device_name(0)}")

# 2. Base model and Dataset for RLHF

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "skt/kogpt2-base-v2"  # "monologg/koelectra-base-v3-discriminator" 

# tokenizer = AutoTokenizer.from_pretrained(model_name)
from transformers import PreTrainedTokenizerFast
base_tokenizer = PreTrainedTokenizerFast.from_pretrained(model_name,
                                                    bos_token='</s>', eos_token='</s>', unk_token='<unk>',
                                                    pad_token='<pad>', mask_token='<mask>')

base_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

- 입력받아 처리할 수 있는 최대 토큰 수 확인

In [ ]:
base_tokenizer.model_max_length

- model.config.n_positions : 모델이 설계상 처리할 수 있는 최대 토큰 길이(최대 시퀀스 길이, context length)

In [ ]:
base_model.config.n_positions

In [ ]:
input_txt = "바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까."

In [ ]:
tokens = base_tokenizer(input_txt).tokens()

input_ids = base_tokenizer(input_txt, return_tensors="pt")["input_ids"].numpy()

In [ ]:
pd.options.display.max_columns = 40
pd.options.display.max_rows = 60
df = pd.DataFrame([tokens, input_ids[0]], index=["kogpt-2_tokens", "Input_IDs"])
df

-  디코딩 성능 확인

In [ ]:
max_length=128
input_ids = base_tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)
output_greedy = base_model.generate(input_ids, max_length=max_length, do_sample=False)
print(base_tokenizer.decode(output_greedy[0]))

시퀀스가 반복되어 출력되는군요.
그리디 서치 디코딩시 발견되는 전형적인 현상입니다.

- 이번엔 빔 서치 디코딩을 사용하고 n-gram 패널티까지 부과해보겠습니다.

In [ ]:
input_ids = base_tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)
output_beam = base_model.generate(input_ids, max_length=max_length, num_beams=10, no_repeat_ngram_size=2,
                             do_sample=False)
print(base_tokenizer.decode(output_beam[0]))

- 이번엔 샘플링 기법까지 추가해 보겠습니다.

In [ ]:
output_beam = base_model.generate(input_ids, max_length=max_length, num_beams=7, no_repeat_ngram_size=2,
                             do_sample=True, temperature=2.0, top_k=50)
print(base_tokenizer.decode(output_beam[0]))

- top_p 샘플링 기법도 사용해보겠습니다.

In [ ]:
output_beam = base_model.generate(input_ids, max_length=max_length, num_beams=7, no_repeat_ngram_size=2,
                             do_sample=True, top_p=0.90)
print(base_tokenizer.decode(output_beam[0]))

## RLHF Dataset 정의

- SFT를 시도할 initial 모델에 쓸 데이터셋

In [ ]:
import json
import os

# data_path_1_SFT = 'KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl'
data_dir = os.path.expanduser('~/work/KoChatGPT/data_kochatgpt')
# data_dir = os.path.expanduser('KoChatGPT-main\data_kochatgpt')
data_path_1_SFT = os.path.join(data_dir, 'kochatgpt_1_SFT.jsonl')

with open(data_path_1_SFT, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

- RM에 사용할 데이터셋

In [ ]:
# data_path_2_RM = 'KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl'
data_dir = os.path.expanduser('~/work/KoChatGPT/data_kochatgpt')
# data_dir = os.path.expanduser('KoChatGPT-main\data_kochatgpt')
data_path_2_RM = os.path.join(data_dir, 'kochatgpt_2_RM.jsonl')

with open(data_path_2_RM, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

- PPO 학습에 쓰일 데이터

In [ ]:
# data_path_3_PPO = 'KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl'
data_dir = os.path.expanduser('~/work/KoChatGPT/data_kochatgpt')
# data_dir = os.path.expanduser('KoChatGPT-main\data_kochatgpt')
data_path_3_PPO = os.path.join(data_dir, 'kochatgpt_3_PPO.jsonl')

with open(data_path_3_PPO, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

# 3. Supervised Fine-Tuning (SFT)

In [35]:
from typing import Optional, Dict, Sequence
from torch.utils.data import Dataset
from dataclasses import dataclass
import logging
import copy

- 모델과 토크나이저를 불러오겠습니다.

In [ ]:
# model = AutoModelForCausalLM.from_pretrained('skt/kogpt2-base-v2')

# tokenizer = AutoTokenizer.from_pretrained(
#     'skt/kogpt2-base-v2', bos_token='</s>', eos_token='</s>',
#     unk_token='</s>', pad_token='</s>',
#     padding_side="right",
#     model_max_length=512,
# )

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "skt/kogpt2-base-v2"

sft_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

from transformers import PreTrainedTokenizerFast

sft_tokenizer = PreTrainedTokenizerFast.from_pretrained(
    "skt/kogpt2-base-v2",
    bos_token="</s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
    mask_token="<mask>",
)

sft_tokenizer.padding_side = "right"
sft_tokenizer.model_max_length = 512

print(sft_tokenizer)

- 모델 인퍼런스 단계에서 사용할 prompt 딕셔너리 템플릿과 SFT 데이터셋 클래스를 정의하겠습니다.

In [ ]:
class SFT_dataset(Dataset):

    def __init__(self, data_path_1_SFT: str, tokenizer: transformers.PreTrainedTokenizer, verbose=False):
        super(SFT_dataset, self).__init__()
        logging.warning("Loading data...")

        pattern_instruction = 'prompt'  # instruction
        pattern_output = 'completion'  # response

        with open(data_path_1_SFT, "r", encoding='utf-8-sig') as json_file:
            list_data_dict = json.load(json_file)

        PROMPT_DICT = {
            "prompt_input": (
                "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
            )
        }

        prompt_input = PROMPT_DICT["prompt_input"]

        sources = []
        for example in list_data_dict:
            tmp = prompt_input.format_map(example)
            sources.append(tmp)

        targets = []
        for example in list_data_dict:
            targets.append(f"{example[pattern_output]}{tokenizer.eos_token}")
        examples = [s + t for s, t in zip(sources, targets)]

        sources_tokenized = self._tokenize_fn(sources, tokenizer)  # source
        examples_tokenized = self._tokenize_fn(examples, tokenizer)  # source + target

        input_ids = examples_tokenized["input_ids"]
        labels = copy.deepcopy(input_ids)
        for label, source_len in zip(labels, sources_tokenized["input_ids_lens"]):
            label[:source_len] = -100 
            # -100 설명
            # -100은 “여기 토큰은 무시해라”는 표시용 마스크 값
            # 프롬프트(instruction) 구간의 라벨을 전부 -100으로 교체
            # PyTorch nn.CrossEntropyLoss는 기본적으로 ignore_index=-100라서,
            # 이 부분은 loss에 기여하지 않고, 해당 토큰들에 대해서는 gradient도 흘러가지 않습니다
            # SFT에서는 보통:
            # 입력: ### Instruction ... ### Response: + 정답 completion
            # 출력(라벨): Response(응답) 부분만 정답으로 보고, 
            # Instruction 부분은 단지 conditioning 컨텍스트로만 사용하고 싶습니다.
            # 그런데 HuggingFace 스타일로 causal LM을 학습할 때는 관례상:
            # input_ids = prompt + completion
            # labels = prompt + completion를 그대로 두고,
            # loss를 계산하고 싶지 않은 구간(여기서는 prompt)을 -100으로 마스킹해서
            # CrossEntropyLoss가 무시하게 만듭니다.
            # 그 결과:
            # 모델은 여전히 프롬프트까지 포함한 전체 시퀀스를 autoregressive하게 인코딩하지만,
            # 업데이트는 completion 토큰들에 대해서만 이루어져서 
            # “질문을 얼마나 잘 예측했는지”는 신경 쓰지 않고,
            # “질문을 보고 답을 얼마나 잘 생성하는지”에만 맞춰집니다.

        data_dict = dict(input_ids=input_ids, labels=labels)

        self.input_ids = data_dict["input_ids"]
        self.labels = data_dict["labels"]
        logging.warning("Loading data done!!: %d"%(len(self.labels)))


    def _tokenize_fn(self, strings: Sequence[str], tokenizer: transformers.PreTrainedTokenizer) -> Dict:
        tokenized_list = [
            tokenizer(
                text,
                return_tensors="pt",
                padding="longest",
                max_length=tokenizer.model_max_length,
                truncation=True,
            )
            for text in strings
        ]
        input_ids = labels = [tokenized.input_ids[0] for tokenized in tokenized_list]
        input_ids_lens = labels_lens = [
            tokenized.input_ids.ne(tokenizer.pad_token_id).sum().item() for tokenized in tokenized_list
        ]
        return dict(
            input_ids=input_ids,
            labels=labels,
            input_ids_lens=input_ids_lens,
            labels_lens=labels_lens,
        )


    def __len__(self):
        return len(self.input_ids)


    def __getitem__(self, i) -> Dict[str, torch.Tensor]:
        return dict(input_ids=self.input_ids[i], labels=self.labels[i])

In [ ]:
@dataclass
class DataCollatorForSupervisedDataset(object):

    tokenizer: transformers.PreTrainedTokenizer

    def __call__(self, instances: Sequence[Dict]) -> Dict[str, torch.Tensor]:
        input_ids, labels = tuple([instance[key] for instance in instances] for key in ("input_ids", "labels"))
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id
        )
        labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value= -100)
        return dict(
            input_ids=input_ids,
            labels=labels,
            attention_mask=input_ids.ne(self.tokenizer.pad_token_id),
        )

- SFT_dataset 클래스를 사용해 훈련셋을 만들고 data collator 인스턴스 생성

In [ ]:
import os
import json

# data_path_1_SFT = 'KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl'
data_dir = os.path.expanduser('~/work/KoChatGPT/data_kochatgpt')
#data_dir = os.path.expanduser('KoChatGPT-main\data_kochatgpt')
data_path_1_SFT = os.path.join(data_dir, 'kochatgpt_1_SFT.jsonl')

train_dataset = SFT_dataset(data_path_1_SFT=data_path_1_SFT, tokenizer=sft_tokenizer)
data_collator = DataCollatorForSupervisedDataset(tokenizer=sft_tokenizer)

print('input : %s'%train_dataset.input_ids[0])
print('output: %s'%train_dataset.labels[0])


- 디코딩하는 함수로 정수토큰들의 인코딩 전 원래 문장의 토크나이징 확인

In [ ]:
# train_dataset.input_ids[0]를 디코딩해보세요.

def inspect_example(tokenizer, input_ids, labels):
    # 1) input_ids 전체 디코딩 (prompt + completion)
    decoded_input = tokenizer.decode(input_ids, skip_special_tokens=False)
    print("=== input_ids (prompt + completion) ===")
    print(decoded_input)
    print()

    # 2) labels에서 -100을 패딩 토큰 id로 되살린 뒤 디코딩
    labels_np = labels.clone()
    ignore_index = -100
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

    labels_np[labels_np == ignore_index] = pad_id
    decoded_labels = tokenizer.decode(labels_np, skip_special_tokens=False)

    print("=== labels (loss 계산 대상 토큰 표시용) ===")
    print(decoded_labels)
    print()

    # 3) completion 부분만 보고 싶으면, -100이 아닌 위치만 모아서 디코딩
    target_token_ids = labels[labels != ignore_index]
    decoded_target = tokenizer.decode(target_token_ids, skip_special_tokens=False)

    print("=== target only (응답 부분만) ===")
    print(decoded_target)
    print()

i = 0
input_ids_0 = train_dataset.input_ids[i]
labels_0    = train_dataset.labels[i]

inspect_example(sft_tokenizer, input_ids_0, labels_0)

- Training arguments를 사용해 trainer 클래스 정의

In [ ]:
# training_args = transformers.TrainingArguments(
#     output_dir="test",
#     overwrite_output_dir=True,
#     num_train_epochs=1,
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
#     warmup_steps=5,
#     prediction_loss_only=True,
#     fp16 = True
#     )
training_args = transformers.TrainingArguments(
    output_dir="test",
    # overwrite_output_dir=True, # 현재 버전에는 없어서 에러 발생
    num_train_epochs=5,  # 10,   # 1,
    logging_steps=100,   # 얼마나 자주 로그를 남길지 설정
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=5,
    prediction_loss_only=True,
    fp16 = True
    )
trainer = transformers.Trainer(
    model=sft_model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset
)

- SFT 훈련 (epoch 1 -> 10 -> 5)

In [ ]:
trainer.train()
sft_model.save_pretrained('models/output_1_SFT')

In [ ]:
# 학습 곡선

import matplotlib.pyplot as plt

# 1. log_history에서 loss 값 추출
history = trainer.state.log_history
train_loss = [log["loss"] for log in history if "loss" in log]
epochs = [log["epoch"] for log in history if "loss" in log]

# 2. 그래프 그리기
plt.figure(figsize=(10, 5))
plt.plot(epochs, train_loss, label="Training Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("SFT Training Loss Curve")
plt.legend()
plt.grid(True)
plt.show()

## 평가 1

이제 문장 생성 능력을 확인하기 위해 빠르게 허깅페이스의 pipleline 클래스를 사용하여 generator를 만들어보겠습니다.

In [ ]:
generator = transformers.pipeline('text-generation', model='models/output_1_SFT', tokenizer=sft_tokenizer)

generation_args = dict(
    num_beams=4,
    repetition_penalty=2.0,
    no_repeat_ngram_size=4,
    eos_token_id=375, # \n
    max_new_tokens=64,
    do_sample=True,
    top_k=50,
    early_stopping=True
)

PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

list_prompt = ['불고기용 고기 한우에요?',
               '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
               '시카고 오헤어 국제공항은 어디에 있어?',
               '오늘 미세먼지 어때?']

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt' : tmp}) for tmp in list_prompt]

list_result = generator(list_prompt, **generation_args)
for prompt, result in zip(list_prompt, list_result):
    print()
    print((result[0]['generated_text']))

## 평가 2

In [ ]:
# 필요한 라이브러리 설치 (한 번만)
# !pip install sacrebleu rouge-score

from sacrebleu import corpus_bleu
from rouge_score import rouge_scorer

def eval_text_gen(references, candidates):
    """
    references: List[str]  (정답 문장들)
    candidates: List[str]  (모델이 생성한 문장들, references와 길이 동일)
    """
    assert len(references) == len(candidates)

    # 1) BLEU (정렬된 전체 코퍼스 기준)
    bleu = corpus_bleu(candidates, [references])
    bleu_score = bleu.score  # 0~100

    # 2) ROUGE-1, ROUGE-L (문장 단위 평균)
    scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=False)
    r1_f, rl_f = 0.0, 0.0
    for ref, cand in zip(references, candidates):
        scores = scorer.score(ref, cand)
        r1_f += scores["rouge1"].fmeasure
        rl_f += scores["rougeL"].fmeasure
    r1_f /= len(references)
    rl_f /= len(references)

    return {
        "bleu": bleu_score,  # 생성 품질 평가
        # "rouge1_f": r1_f,  # 정답에 있는 중요한 내용, 표현을 얼마나 포함했나?
        # "rougeL_f": rl_f,
    }


In [ ]:
import re

max_length=128

list_prompt = ['불고기용 고기 한우에요?',
               '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
               '시카고 오헤어 국제공항은 어디에 있어?',
               '오늘 미세먼지 어때?']

for i in range(len(list_prompt)):

    # prompt
    in_str = [list_prompt[i]]

    # 기존 모델 
    input_ids = base_tokenizer(list_prompt[i], return_tensors="pt")["input_ids"].to(device)
    output_beam = base_model.generate(input_ids, max_length=max_length, num_beams=7, no_repeat_ngram_size=2,
                                      do_sample=True, top_p=0.90)
    output = base_tokenizer.decode(output_beam[0], skip_special_tokens=True)
    prompt_len = len(base_tokenizer(list_prompt[i], return_tensors="pt")["input_ids"][0])
    generated = output[prompt_len:]
    base_model_sentence = [generated.split('\n')[1]]

    base_score = eval_text_gen(in_str, base_model_sentence)


    
    # SFT (먼저 실행 결과 이용)
    temp_str2 = list_result[i][0]['generated_text']
    temp_str2 = temp_str2.split('### Response(응답):')[1]
    first_sentence = re.split(r'[.。]', temp_str2)[0].strip() + '.'
    clean_sentence = [first_sentence.strip().lstrip("'")]
    
    sft_score = eval_text_gen(in_str, clean_sentence)

    print("-" * 50)
    print("No: ", i+1)
    print("Prompt: ", in_str)
    print("Base Model Response: ", base_model_sentence)
    print("Base Model BLEU: ", base_score) 
    print("SFT Model Response: ", clean_sentence)
    print("SFT Model BLEU: ", sft_score) 


- 메모리 관리를 위해 캐시를 비우고 넘어가겠습니다.

In [ ]:
torch.cuda.empty_cache()

# 4. Reward Model (RM)

In [36]:
from chatgpt.dataset import RewardDataset
from chatgpt.models.base import RewardModel
from chatgpt.trainer.strategies import NaiveStrategy
from chatgpt.trainer.rm import RewardModelTrainer

from transformers.models.gpt2.configuration_gpt2 import GPT2Config
from transformers.models.gpt2.modeling_gpt2 import GPT2Model

import torch.nn as nn

import random

- GPTRM_custom 이라는 이름으로 클래스 선언

In [37]:
class GPTRM_custom(RewardModel):

    def __init__(self,
                 pretrained: Optional[str] = None,
                 config: Optional[GPT2Config] = None,
                 checkpoint: bool = False,
                 lora_rank: int = 0,
                 lora_train_bias: str = 'none',
                 tokenizer=None) -> None:
        if pretrained is not None:
            model = GPT2Model.from_pretrained(pretrained)
            model.resize_token_embeddings(len(tokenizer))
        elif config is not None:
            model = GPT2Model(config)
        else:
            model = GPT2Model(GPT2Config())
        if checkpoint:
            model.gradient_checkpointing_enable()

        value_head = nn.Linear(model.config.n_embd, 1)
        # model.config.n_embd: GPT-2의 내부 은닉 상태 벡터의 차원(예: 768, 1024 등).
        # 1: 최종적으로 하나의 보상 값(스칼라)을 출력하기 위한 차원.

        super().__init__(model, value_head, lora_rank, lora_train_bias)

        if pretrained is not None:
            self.model = model
            self.pretrained = pretrained


    def save_pretrained(self, dir):
        if self.pretrained is not None:
            self.model.save_pretrained(dir)

In [47]:
# model = AutoModelForCausalLM.from_pretrained('skt/kogpt2-base-v2')
# tokenizer = AutoTokenizer.from_pretrained(
#     'skt/kogpt2-base-v2', bos_token='</s>', eos_token='</s>', unk_token='</s>', pad_token='</s>',
#     padding_side="right",
#     model_max_length=512,
# )

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "skt/kogpt2-base-v2"

rm_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

from transformers import PreTrainedTokenizerFast

rm_tokenizer = PreTrainedTokenizerFast.from_pretrained(
    "skt/kogpt2-base-v2",
    bos_token="</s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
    mask_token="<mask>",
)

rm_tokenizer.padding_side = "right"
rm_tokenizer.model_max_length = 512

with NaiveStrategy().model_init_context():
        rm_model = GPTRM_custom(pretrained='skt/kogpt2-base-v2', lora_rank=0, tokenizer=rm_tokenizer).cuda()

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
lm_head.weight                          | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


- RM 훈련시킬 때 사용할 ranking dataset 생성

In [48]:
import os
import json

# data_path_2_RM = 'KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl'
data_dir = os.path.expanduser('~/work/KoChatGPT/data_kochatgpt')
# data_dir = os.path.expanduser('KoChatGPT-main\data_kochatgpt')
data_path_2_RM = os.path.join(data_dir, 'kochatgpt_2_RM.jsonl')

with open(data_path_2_RM, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

total_data_ranking2chosen = []
for tmp in list_data_dict:
    one_data_ranking2chosen = []

    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][0] < tmp['ranking'][1]:
        data['chosen'] = tmp['completion_0']
        data['rejected'] = tmp['completion_1']
    else:
        data['chosen'] = tmp['completion_1']
        data['rejected'] = tmp['completion_0']
    one_data_ranking2chosen.append(data)

    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][0] < tmp['ranking'][2]:
        data['chosen'] = tmp['completion_0']
        data['rejected'] = tmp['completion_2']
    else:
        data['chosen'] = tmp['completion_2']
        data['rejected'] = tmp['completion_0']
    one_data_ranking2chosen.append(data)

    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][1] < tmp['ranking'][2]:
        data['chosen'] = tmp['completion_1']
        data['rejected'] = tmp['completion_2']
    else:
        data['chosen'] = tmp['completion_2']
        data['rejected'] = tmp['completion_1']
    one_data_ranking2chosen.append(data)

    total_data_ranking2chosen.extend(one_data_ranking2chosen)

print('before data num: %d'%(len(list_data_dict)))
print('after  data num: %d'%(len(total_data_ranking2chosen)))
print('data example: \n%s'%total_data_ranking2chosen[45])

before data num: 10220
after  data num: 30660
data example: 
{'prompt': '애플은 리사를 어떻게 처리했어', 'chosen': '애플이 누구인지 명확히 알 수 없어서, 리사가 누구인지와 어떤 상황에서 처리되었는지에 대한 추가적인 정보가 필요합니다. 따라서, 보다 정확한 답변을 제공할 수 없습니다.', 'rejected': '애플은 리사를 위해 고객 서비스 부서에서 고객 다양한 컴퓨터 관련 문제에 대해 응답하는 데 필요한 모든 지원을 제공했습니다. 사용자가 하드웨어 문제를 경험할 때, 전문가들은 필요한 수리(수리, 추가 부품 제공, 소프트웨어 업그레이드 등)을 제공해 드릴 수 있습니다. 또한, 사용자가 사용 방법 문제나 기타 문제를 경험할 때, 대화 상대로 사용자를 지원할 수 있는 전문 고객 서비스 직원들이 사용자에게 상담하고 도움을 주는 데 도움이 될 수 있는 정보를 제공합니다. 또한, 인터넷에서 제공되는 정보를 통해 문제를 해결하거나 고객 서비스 웹 사이트를 통해 자신의 문제를 진단할 수 있도록 하는 등 다양한 방법으로 리사를 처리해 왔습니다.'}


In [49]:
class PairWiseLoss(nn.Module):

    def forward(self, chosen_reward: torch.Tensor, reject_reward: torch.Tensor) -> torch.Tensor:
        probs = torch.sigmoid(chosen_reward - reject_reward)
        log_probs = torch.log(probs)
        loss = -log_probs.mean()
        return loss
"""
- probs = torch.sigmoid(chosen_reward - reject_reward)
  chosen_reward - reject_reward는 선택된(chosen) 보상과 거부된(reject) 보상 간의 차이를 계산하는 연산입니다.
  이 차이가 클수록 선택된 샘플이 거부된 샘플보다 높은 보상을 받았다는 것을 의미하죠.
  여기서 `torch.sigmoid(chosen_reward - reject_reward)`를 적용하면,
  두 보상의 차이를 확률 값(0과 1 사이)으로 변환하게 됩니다.
  이 확률 값은 선택된 샘플이 거부된 샘플보다 더 좋은 결과를 낼 확률로 해석할 수 있습니다.
-loss = -log_probs.mean() 코드는 선택된 샘플이 거부된 샘플보다 더 좋은 결과를 내는
  log 확률을 최대화하는 방향으로 작동합니다.
"""

'\n- probs = torch.sigmoid(chosen_reward - reject_reward)\n  chosen_reward - reject_reward는 선택된(chosen) 보상과 거부된(reject) 보상 간의 차이를 계산하는 연산입니다.\n  이 차이가 클수록 선택된 샘플이 거부된 샘플보다 높은 보상을 받았다는 것을 의미하죠.\n  여기서 `torch.sigmoid(chosen_reward - reject_reward)`를 적용하면,\n  두 보상의 차이를 확률 값(0과 1 사이)으로 변환하게 됩니다.\n  이 확률 값은 선택된 샘플이 거부된 샘플보다 더 좋은 결과를 낼 확률로 해석할 수 있습니다.\n-loss = -log_probs.mean() 코드는 선택된 샘플이 거부된 샘플보다 더 좋은 결과를 내는\n  log 확률을 최대화하는 방향으로 작동합니다.\n'

In [50]:
total_data_ranking2chosen = []

for tmp in list_data_dict:
     prompt = tmp['prompt']
     ranking = tmp['ranking']

     for index in range(1, len(ranking)):
         n = ranking[0]
         m = ranking[index]

         data = {
             'prompt': prompt,
             'chosen': tmp['completion_{}'.format(n)],
             'rejected': tmp['completion_{}'.format(m)]
         }

         total_data_ranking2chosen.append(data)

- 완성한 ranking dataset을 shuffle한 후 훈련셋을 만들어보겠습니다.

빠르게 돌려보기 위해 전체 데이터중 일부만 학습하도록 하겠습니다.

In [51]:
import random
random.seed(230319)
random.shuffle(total_data_ranking2chosen)
print(total_data_ranking2chosen[45])

{'prompt': '멋있게 잊어 주자', 'chosen': '가끔은 일어나서 눈 앞의 것들을 확실하게 정리하는 것도 좋을 때가 있죠. 그렇게 해서 지우는 것이 더 나은 경우도 있으니까요. 그리고 그렇게 해서 잊어버리면 조금 더 나은 상황으로 다가갈 수 있을 거라 생각합니다. 그러니 마음을 비워놓고 새로운 시작을 해보세요. 더 나은 날들이 오길 바랄게요!', 'rejected': '멋있다 공개 개량\n\n잊어 개량\n\n주자 개량\n\n멋있다 공개 개량\n\n잊어 개량\n\n주자 개량'}


In [52]:
train_data = total_data_ranking2chosen[:1000]
eval_data = total_data_ranking2chosen[1000:1200]

print(len(train_data))
print(len(eval_data))

train_dataset = RewardDataset(train_data, rm_tokenizer, 512)
eval_dataset = RewardDataset(eval_data, rm_tokenizer, 512)

"""
RewardDataset 클래스 기능
- 입력 데이터 처리
각 데이터 항목은 prompt, chosen, rejected와 같은 키를 가지며,
이 클래스는 해당 텍스트들을 받아서 모델 학습에 적합한 형식으로 변환합니다.

- 토큰화(tokenization)
주어진 토크나이저(tokenizer)를 사용하여 텍스트를 토큰 ID 시퀀스로 변환하고,
최대 길이(여기서는 512)로 잘라내거나 패딩(padding)을 적용합니다.

- 데이터셋 생성
전처리된 데이터를 PyTorch의 Dataset 형식(예: torch.utils.data.Dataset)으로 만들어,
Trainer 등에서 배치 단위로 데이터를 불러올 수 있도록 합니다.
"""

1000
200


  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

'\nRewardDataset 클래스 기능\n- 입력 데이터 처리\n각 데이터 항목은 prompt, chosen, rejected와 같은 키를 가지며,\n이 클래스는 해당 텍스트들을 받아서 모델 학습에 적합한 형식으로 변환합니다.\n\n- 토큰화(tokenization)\n주어진 토크나이저(tokenizer)를 사용하여 텍스트를 토큰 ID 시퀀스로 변환하고,\n최대 길이(여기서는 512)로 잘라내거나 패딩(padding)을 적용합니다.\n\n- 데이터셋 생성\n전처리된 데이터를 PyTorch의 Dataset 형식(예: torch.utils.data.Dataset)으로 만들어,\nTrainer 등에서 배치 단위로 데이터를 불러올 수 있도록 합니다.\n'

- 데이터셋이 잘 만들어졌는지 하나를 뽑아 확인해봅시다.

In [53]:
idx = 1
print('#'*70)
print('## prompt ##')
print(train_data[idx]['prompt'])
print('#'*70)
print('## chosen ##')
print(train_data[idx]['chosen'])
print('#'*70)
print('## rejected ##')
print(train_data[idx]['rejected'])

######################################################################
## prompt ##
가방 같은 것도 수선해줘요?
######################################################################
## chosen ##
네, 가방도 수선이 가능합니다. 다만 사용하고 있는 가방의 종류와 손상 정도에 따라 수선 방법과 비용이 달라질 수 있습니다. 수선 전에는 꼭 전문가의 상담을 받아보시는 것이 좋습니다.
######################################################################
## rejected ##
다행히 가방을 수선해 드릴 수 있습니다. 다만 보다 정확한 예상 가격을 알기 위해 약간의 정보가 필요합니다. 먼저 가방에 대한 정보를 알려주시면 저희는 가격 협의를 위해 가방을 보고 다시 연락드리겠습니다.


- 마지막으로 RM을 학습해 보겠습니다. (1 epoch)

In [54]:
# RM은 과적합 방지를 위해 epoch : 1 ~ 3 으로 권장
# 학습이 많으면 overfitting 가능성 (모델이 0, 1, 2에서 선택하므로 많은 학습 필요 없음)

trainer = RewardModelTrainer(model=rm_model,
                             strategy=NaiveStrategy(),
                             optim=torch.optim.Adam(rm_model.parameters(), lr=5e-5),
                             train_dataset=train_dataset,
                             eval_dataset=eval_dataset,
                             batch_size=4,
                             max_epochs=2   # 1
)

In [55]:
trainer.fit(use_lora=0)

rm_model.save_pretrained('models/output_2_RM')

Train epoch:   0%|          | 0/2 [00:00<?, ?it/s]

Train step of epoch 0:   0%|          | 0/250 [00:00<?, ?it/s]

Train step of epoch 1:   0%|          | 0/250 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 평가 (reward score)
RM 학습이 잘 되었는지 확인해보기 위해 임의의 문장을 입력한 후 적절한 reward score를 출력하는지 살펴보도록 하겠습니다.

In [ ]:
def inference_RM(input_text):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').cuda()
    output = model(input_ids)
    output_reward = output.cpu().detach().numpy()[0]

    print('input: %s\nreward score: %.1f'%(input_text, output_reward))

    return output_reward

input_text = '인공지능은 똥멍청이 입니다'
output_reward = inference_RM(input_text=input_text)



In [ ]:
input_text = '인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다.'

output_reward = inference_RM(input_text=input_text)

In [ ]:
input_text = "인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다. AI는 현대적인 컴퓨팅 혁신에서 중추적인 역할을 하며 개인과 비즈니스의 가치를 창출합니다. 예를 들어 광학 문자 인식(OCR)은 AI를 사용해 이미지 및 문서에서 텍스트 및 데이터를 추출하고, 구조화되지 않은 콘텐츠를 비즈니스에 바로 사용할 수 있게 만들고, 유용한 정보를 창출합니다."

output_reward = inference_RM(input_text=input_text)

In [ ]:
input_text = "인공지능은 일반적으로 인간의 지능이 필요하거나 인간이 분석할 수 있는 것보다 규모가 큰 데이터를 포함하는 방식으로 추론, 학습 및 행동할 수 있는 컴퓨터 및 기계를 구축하는 것과 관련된 과학 분야입니다. AI는 컴퓨터 공학, 데이터 분석 및 통계, 하드웨어 및 소프트웨어 엔지니어링, 언어학, 신경 과학은 물론 철학과 심리학을 포함하여 여러 학문을 포괄하는 광범위한 분야입니다. 비즈니스의 운영 수준에서 AI는 주로 머신러닝과 딥 러닝을 기반으로 하는 기술 모음으로, 데이터 분석, 예상 및 예측, 객체 분류, 자연어 처리, 추천, 지능형 데이터 가져오기 등을 수행할 수 있습니다."

output_reward = inference_RM(input_text=input_text)

### RM 비교 테스트

In [ ]:
# 비교 테스트 예시
prompts = [
    ("인공지능은 정말 똑똑해!", "인공지능은 똥멍청이야"),
    ("오헤어 공항은 시카고에 있어.", "오헤어 공항은 LA에 있어."),
    ("개포주공아파트는 27 단지로 이루어져 있습니다.", "개포주공아파트는 하늘이가 살고 있습니다.")
]

for chosen, rejected in prompts:
    c_score = inference_RM(chosen)
    r_score = inference_RM(rejected)
    print(f"변별력(Gap): {c_score - r_score:.2f} (양수여야 정상)")
    print()

- 여기서도 메모리 관리를 위해 한 번더 캐시를 비우고 넘어가겠습니다.

In [ ]:
torch.cuda.empty_cache()

# 5. Proximal Policy Optimization (PPO)

In [ ]:
from chatgpt.models.gpt import GPTActor, GPTCritic
from chatgpt.trainer import PPOTrainer
from chatgpt.trainer.strategies import NaiveStrategy
from transformers import PreTrainedTokenizerFast
from copy import deepcopy

In [ ]:
# device = "cuda" if torch.cuda.is_available() else "cpu"
# model_name = "skt/kogpt2-base-v2"

# model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# with NaiveStrategy().model_init_context():
#         model = GPTRM_custom(pretrained='skt/kogpt2-base-v2', lora_rank=0, tokenizer=tokenizer).cuda()

# with NaiveStrategy().model_init_context():
#     actor = GPTActor(pretrained='models/output_1_SFT', lora_rank=0).to(torch.cuda.current_device())
#     critic = GPTCritic(pretrained='models/output_2_RM', lora_rank=0).to(torch.cuda.current_device())
#     tokenizer = AutoTokenizer.from_pretrained(
#         'skt/kogpt2-base-v2', bos_token='</s>', eos_token='</s>', unk_token='</s>', pad_token='</s>',
#         padding_side="right",
#         model_max_length=512
#     )
#     initial_model = deepcopy(actor)
#     reward_model = RewardModel(deepcopy(critic.model), deepcopy(critic.value_head)).to(torch.cuda.current_device())

with NaiveStrategy().model_init_context():
    actor = GPTActor(pretrained='models/output_1_SFT', lora_rank=0).to(torch.cuda.current_device())
    critic = GPTCritic(pretrained='models/output_2_RM', lora_rank=0).to(torch.cuda.current_device())
    
    tokenizer = PreTrainedTokenizerFast.from_pretrained(
        "skt/kogpt2-base-v2",
        bos_token="</s>",
        eos_token="</s>",
        unk_token="<unk>",
        pad_token="<pad>",
        mask_token="<mask>",
    )
    
    tokenizer.padding_side = "right"
    tokenizer.model_max_length = 512
    
    initial_model = deepcopy(actor)
    reward_model = RewardModel(deepcopy(critic.model), deepcopy(critic.value_head)).to(torch.cuda.current_device())

- 모델학습에 사용할 옵티마이저와 모델을 준비합니다.

In [ ]:
actor_optim = torch.optim.Adam(actor.parameters(), lr=5e-6)
critic_optim = torch.optim.Adam(critic.parameters(), lr=5e-6)

In [ ]:
import os
import json

# data_path_3_PPO = 'KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl'
# data_dir = os.path.expanduser('~/work/KoChatGPT/data_kochatgpt')
data_dir = os.path.expanduser('KoChatGPT-main\data_kochatgpt')
data_path_3_PPO = os.path.join(data_dir, 'kochatgpt_3_PPO.jsonl')

with open(data_path_3_PPO, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)
    list_prompt = [tmp['prompt'] for tmp in list_data_dict]

def tokenize_fn(texts):
    batch = tokenizer(texts, return_tensors='pt', max_length=96, padding=True, truncation=True)
    return {k: v.cuda() for k, v in batch.items()}

In [ ]:
(actor, actor_optim), (critic, critic_optim), reward_model, initial_model = NaiveStrategy().prepare( \
    (actor, actor_optim), (critic, critic_optim), reward_model, initial_model)

- PPO 학습에 쓸 데이터를 불러와 토크나이징 해줍니다.

In [ ]:
print(tokenize_fn('It takes something more than intelligence to act intelligently.'))

In [ ]:
len(list_prompt)

- PPO는 별도의 PPOTrainer 클래스를 설계하여 학습시켜줘야 합니다. (1 opoch)

In [ ]:
trainer = PPOTrainer(NaiveStrategy(),
                     actor,
                     critic,
                     reward_model,
                     initial_model,
                     actor_optim,
                     critic_optim,
                     max_epochs=1,
                     train_batch_size=16,  # 8,
                     tokenizer=tokenize_fn,
                     max_length=128,
                     do_sample=True,
                     temperature=1.0,
                     top_k=50,
                     pad_token_id=tokenizer.pad_token_id,
                     eos_token_id=tokenizer.eos_token_id)


In [ ]:
#
# max_epochs=1: 버퍼에 쌓인 데이터를 한 번의 업데이트 루프에서 몇 번 반복해서 학습할지를 의미합니다.
#                PPO에서는 보통 1~10 사이를 사용하며, 1이면 수집된 데이터를 딱 한 번 보고 버립니다.
# num_episodes=10: 전체 학습의 큰 사이클(Episode)을 10번 반복하겠다는 뜻입니다.
# max_timesteps=3: 한 에피소드 내에서 환경(Experience 수집)과 상호작용하는 횟수입니다.
# update_timesteps=3: 몇 번의 타임스텝마다 모델을 업데이트할지 결정합니다.
# PPO 트레이너의 fit 로직은 보통 다음과 같이 동작합니다:
# 데이터(Experience)를 수집합니다.
#수집된 데이터가 update_timesteps만큼 쌓이면 업데이트를 시작합니다.
# 이때 max_epochs만큼 반복해서 가중치를 수정합니다.
# 총 업데이트 횟수 = num_episodes (10) × (max_timesteps / update_timesteps) × max_epochs (1)
# 즉, 총 10번의 가중치 업데이트가 일어납니다.
    
stats = trainer.fit(list_prompt,
            num_episodes=20,     # 10,
            max_timesteps=10,    # 3,
            update_timesteps=10  # 3
        )
actor.model.save_pretrained('models/output_3_PPO')

print(stats)
# # stats 안에 있는 reward와 kl 값을 직접 출력
#     print(f"Episode [{episode+1}/10] Reward: {stats['reward']:.4f}, KLD: {stats['kl']:.4f}")

# actor.model.save_pretrained('models/output_3_PPO')

## 평가
RLHF가 적용된 koGPT-2의 생성능력을 확인해볼까요?

In [ ]:
def generation(input_text, model):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').to(
        torch.cuda.current_device())
    outputs = model.generate(input_ids,
                             max_length=250,
                             do_sample=True,
                             top_k=50,    # 상위 k개의 단어 후보만 고려
                             top_p=0.95,  # 확률 누적 기준으로 상위 p%의 단어들만 고려
                             no_repeat_ngram_size=2,
                             num_return_sequences=1)
    output = tokenizer.batch_decode(outputs[0], skip_special_tokens=True)[0]
    print()
    print(output)
    return output

PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

list_prompt = [
    '불고기용 고기 한우에요?',
    '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
    '시카고 오헤어 국제공항은 어디에 있어',
    '오늘 미세먼지 어때?']

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt': tmp}) for tmp in list_prompt]

for input_text in list_prompt:
    output = generation(input_text, actor)


In [ ]:
from sacrebleu import corpus_bleu
from rouge_score import rouge_scorer

def eval_text_gen2(references, candidates):
    """
    references: List[str]  (정답 문장들)
    candidates: List[str]  (모델이 생성한 문장들, references와 길이 동일)
    """
    assert len(references) == len(candidates)

    # 1) BLEU (정렬된 전체 코퍼스 기준)  # 생성 품질 평가
    bleu = corpus_bleu(candidates, [references])
    bleu_score = bleu.score  # 0~100

    # 2) ROUGE-1, ROUGE-L (문장 단위 평균)  # 정답에 있는 중요한 내용, 표현을 얼마나 포함했나?
    scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=False)
    r1_f, rl_f = 0.0, 0.0
    for ref, cand in zip(references, candidates):
        scores = scorer.score(ref, cand)
        r1_f += scores["rouge1"].fmeasure
        rl_f += scores["rougeL"].fmeasure
    r1_f /= len(references)
    rl_f /= len(references)

    return bleu_score, r1_f, rl_f


def generation2(input_text, model):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').to(torch.cuda.current_device())
    
    outputs = model.generate(
        input_ids,
        max_length=250,
        do_sample=True,
        top_k=1,     # 상위 k개의 단어 후보만 고려
        top_p=0.95,  # 확률 누적 기준으로 상위 p%의 단어들만 고려
        no_repeat_ngram_size=2,
        num_return_sequences=1
    )
    output = tokenizer.batch_decode(outputs[0], skip_special_tokens=True)[0]
    
    out_temp = output.split('### Response(응답):')[1]
    out_part = out_temp.split(".", 1)[0]
    out_clean = [out_part.strip().lstrip("'")]
    
    in_temp = input_text.split('### Instruction(명령어):\n')[1]
    in_str = [in_temp.split("\n", 1)[0]]
    

    scores, _, _ = eval_text_gen2(in_str, out_clean)

    print("-" * 50)
    print("Prompt: ", in_str)
    print("Response: ", out_clean)
    print("BLEU: ", scores) 


PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

list_prompt = [
    '불고기용 고기 한우에요?',
    '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
    '시카고 오헤어 국제공항은 어디에 있어',
    '오늘 미세먼지 어때?']

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt': tmp}) for tmp in list_prompt]

for input_text in list_prompt:
    output = generation2(input_text, actor)
